In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import googlenet, GoogLeNet_Weights
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import os

import torch._dynamo
torch._dynamo.config.suppress_errors = True


In [10]:

# Device & performance settings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

Using device: cuda


In [11]:

# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")
print("Class names:", dataset.classes[:10], "...")

# Stratified split
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count())

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

print("DataLoaders ready")

Total images: 100000
Number of classes: 200
Class names: ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750'] ...
Train: 85000, Val: 10000, Test: 5000
✅ DataLoaders ready


In [12]:

# Model (GoogLeNet)

weights = GoogLeNet_Weights.DEFAULT
model = googlenet(weights=weights, aux_logits=True)

num_classes = 200
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.aux1.fc2 = nn.Linear(model.aux1.fc2.in_features, num_classes)
model.aux2.fc2 = nn.Linear(model.aux2.fc2.in_features, num_classes)

model = model.to(device, memory_format=torch.channels_last)

if hasattr(torch, "compile"):
    model = torch.compile(model, mode="max-autotune")

/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torchvision/models/googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


In [13]:
print(model.aux1)

InceptionAux(
  (conv): BasicConv2d(
    (conv): Conv2d(512, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (fc1): Linear(in_features=2048, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=200, bias=True)
  (dropout): Dropout(p=0.7, inplace=False)
)


In [14]:

# Loss, Optimizer & Scheduler

criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=1e-4, nesterov=True)

epochs = 90  # <-- define here before using in scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


In [15]:

# Training & Evaluation

def evaluate_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if isinstance(outputs, tuple):  # googlenet returns (main, aux1, aux2) during training
                outputs = outputs[0]
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=90, patience=10):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    
    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type=="cuda")):
                outputs = model(images)
                if isinstance(outputs, tuple):   # GoogLeNet has aux outputs
                    main_out, aux1_out, aux2_out = outputs
                    loss = (criterion(main_out, labels) +
                            0.3 * criterion(aux1_out, labels) +
                            0.3 * criterion(aux2_out, labels))
                else:
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = (main_out if isinstance(outputs, tuple) else outputs).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        val_acc = evaluate_model(model, val_loader, device)
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {running_loss/total:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        #  Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "googlenet_best.pth")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f" Early stopping triggered at epoch {epoch+1}")
            break

        #  Step the scheduler
        scheduler.step()

In [16]:

# Run Training

epochs = 20
train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs)

AUTOTUNE convolution(128x3x224x224, 64x3x7x7)
  convolution 2.0972 ms 100.0%
  triton_convolution_1568 2.7433 ms 76.4%
  triton_convolution_1569 2.9860 ms 70.2%
  triton_convolution_1570 3.2174 ms 65.2%
  triton_convolution_1565 3.6710 ms 57.1%
  triton_convolution_1567 5.1205 ms 41.0%
  triton_convolution_1566 8.3507 ms 25.1%
SingleProcess AUTOTUNE takes 5.9698 seconds
AUTOTUNE mm(401408x64, 64x64)
  triton_mm_1579 0.0840 ms 100.0%
  triton_mm_1572 0.0870 ms 96.5%
  triton_mm_1573 0.0870 ms 96.5%
  triton_mm_1571 0.0881 ms 95.3%
  triton_mm_1574 0.0881 ms 95.3%
  triton_mm_1575 0.0881 ms 95.3%
  triton_mm_1578 0.0911 ms 92.1%
  mm 0.0922 ms 91.1%
  triton_mm_1580 0.0932 ms 90.1%
  triton_mm_1576 0.1188 ms 70.7%
SingleProcess AUTOTUNE takes 4.5872 seconds
AUTOTUNE convolution(128x64x56x56, 192x64x3x3)
  convolution 0.4536 ms 100.0%
  triton_convolution_1586 0.7393 ms 61.4%
  triton_convolution_1584 0.8028 ms 56.5%
  triton_convolution_1589 0.8366 ms 54.2%
  triton_convolution_1587 1.00

Epoch [1/20] | Train Loss: 5.6001 | Train Acc: 40.49% | Val Acc: 39.60%
Epoch [2/20] | Train Loss: 4.6323 | Train Acc: 54.50% | Val Acc: 40.63%
Epoch [3/20] | Train Loss: 4.2803 | Train Acc: 59.81% | Val Acc: 50.95%
Epoch [4/20] | Train Loss: 4.0721 | Train Acc: 63.08% | Val Acc: 53.41%
Epoch [5/20] | Train Loss: 3.9190 | Train Acc: 65.52% | Val Acc: 49.09%
Epoch [6/20] | Train Loss: 3.7973 | Train Acc: 67.65% | Val Acc: 53.54%
Epoch [7/20] | Train Loss: 3.7168 | Train Acc: 69.08% | Val Acc: 51.96%
Epoch [8/20] | Train Loss: 3.6424 | Train Acc: 70.16% | Val Acc: 47.23%
Epoch [9/20] | Train Loss: 3.5808 | Train Acc: 71.57% | Val Acc: 52.80%
Epoch [10/20] | Train Loss: 3.5275 | Train Acc: 72.18% | Val Acc: 48.44%
Epoch [11/20] | Train Loss: 3.4711 | Train Acc: 73.20% | Val Acc: 54.71%
Epoch [12/20] | Train Loss: 3.4175 | Train Acc: 74.33% | Val Acc: 48.29%
Epoch [13/20] | Train Loss: 3.3881 | Train Acc: 74.73% | Val Acc: 52.06%
Epoch [14/20] | Train Loss: 3.3527 | Train Acc: 75.35% | Val

In [17]:

# Save Weights

torch.save(model.state_dict(), "googlenet_weights.pth")
print("Saved GoogLeNet weights")

# Final Test Accuracy

test_acc = evaluate_model(model, test_loader, device)
print(f"Final Test Accuracy: {test_acc:.2f}%")

💾 Saved GoogLeNet weights
🎯 Final Test Accuracy: 52.78%
